# 4주차 ③ 로지스틱 회귀 — 실습 7~8

**목표**: 출력에 시그모이드를 씌우고 손실을 BCE로 바꾸면 **같은 구조가 분류가 된다**는 것을 확인하고,
학습된 결정 경계를 그려 **직선 하나의 한계**를 눈으로 본다.

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.
> 아니면 커널명을 눌러 `Python (dl2026)` 을 선택하세요. (2주차 실습 8에서 다룬 그것입니다.)

> **모델 구조는 회귀와 같습니다.** `nn.Linear` 그대로입니다.
> 바뀌는 것은 **출력을 어떻게 해석하는가(시그모이드)** 와 **어떻게 틀린 정도를 재는가(BCE)** 뿐입니다.

## 실습 7 — 이진 분류

In [ ]:
# 셀 1 — 두 부류 데이터
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

torch.manual_seed(0)
N = 100
c0 = torch.randn(N, 2) + torch.tensor([-2.0, -2.0])    # 부류 0
c1 = torch.randn(N, 2) + torch.tensor([ 2.0,  2.0])    # 부류 1

X = torch.cat([c0, c1])                                 # (200, 2)
y = torch.cat([torch.zeros(N, 1), torch.ones(N, 1)])    # (200, 1)
print("X.shape :", X.shape, "| y.shape :", y.shape)

plt.scatter(c0[:, 0], c0[:, 1], s=12, label="부류 0")
plt.scatter(c1[:, 0], c1[:, 1], s=12, label="부류 1")
plt.legend(); plt.title("이진 분류 데이터"); plt.show()

입력이 **2차원**이 되었으므로 모델은 `nn.Linear(2, 1)` — 입력 2개, 출력 1개(로짓)입니다.
`(200,2) @ (2,1) → (200,1)` 을 스스로 검산해 보세요.

> `y` 의 shape 이 `(200, 1)` 인 것에 주의하세요. `(200,)` 이면 `BCEWithLogitsLoss` 에서
> 브로드캐스팅 경고가 나거나 엉뚱하게 계산됩니다.

In [ ]:
# 셀 2 — 학습 (루프는 실습 6과 똑같다)
model = nn.Linear(2, 1)                       # ★ 구조는 회귀와 같다
loss_fn = nn.BCEWithLogitsLoss()              # ★ 손실만 바뀐다
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history = []
for epoch in range(300):
    logit = model(X)                          # ① 순전파 (시그모이드 없이!)
    loss = loss_fn(logit, y)                  # ② 손실

    optimizer.zero_grad()
    loss.backward()                           # ③ 역전파
    optimizer.step()                          # ④ 갱신

    history.append(loss.item())
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")

실습 6과 **줄 단위로 비교**해 보세요. 다른 것은 두 곳뿐입니다.

```
nn.Linear(1, 1)      →  nn.Linear(2, 1)          입력이 2차원이라서
nn.MSELoss()         →  nn.BCEWithLogitsLoss()   분류라서
```

**학습 루프는 한 글자도 안 바뀌었습니다.**

> **`BCEWithLogitsLoss` 에 시그모이드를 따로 붙이지 않는 이유**: 이 함수가 **안에서 처리**합니다.
> 따로 붙이면 **두 번 씌우는 것**이 되어 학습이 이상해집니다.
> 나눠서 계산하면 확률이 0이나 1에 가까울 때 `log(0)` 이 되어 `inf` 가 나오는데,
> 합쳐 계산하면 그 문제가 없습니다 — **수치 안정성**. 중간고사 출제 지점입니다.

In [ ]:
# 셀 3 — 정확도 확인
with torch.no_grad():                          # ★ 평가에는 그래프가 필요 없다
    prob = torch.sigmoid(model(X))             # ★ 여기서만 시그모이드를 씌운다
    pred = (prob > 0.5).float()
    acc = (pred == y).float().mean()

print(f"정확도 : {acc.item()*100:.1f}%")
print("\n앞 5개 확인")
for i in [0, 1, 100, 101, 102]:
    print(f"  확률 {prob[i].item():.3f} → 예측 {int(pred[i].item())} | 정답 {int(y[i].item())}")

정확도가 100%에 가까운 것은 **데이터를 그렇게 만들었기 때문**입니다.
두 덩어리가 확실히 떨어져 있으면 직선 하나로 갈라집니다.
6주차에 **정확도만 보면 안 되는 이유**(불균형 데이터, 정밀도·재현율)를 배웁니다.

## 실습 8 — 결정 경계 시각화

로짓이 0이 되는 지점이 경계입니다. `w1·x1 + w2·x2 + b = 0` → `x2 = -(w1·x1 + b) / w2`

In [ ]:
# 셀 4 — 결정 경계
with torch.no_grad():
    w1, w2 = model.weight[0]
    bias = model.bias[0]

xs = torch.linspace(-6, 6, 100)
ys = -(w1 * xs + bias) / w2                   # 로짓 = 0 인 선

plt.figure(figsize=(6, 5))
plt.scatter(c0[:, 0], c0[:, 1], s=12, label="부류 0")
plt.scatter(c1[:, 0], c1[:, 1], s=12, label="부류 1")
plt.plot(xs, ys, color="red", linewidth=2, label="결정 경계 (로짓 = 0)")
plt.xlim(-6, 6); plt.ylim(-6, 6); plt.legend()
plt.title("학습된 결정 경계"); plt.show()

빨간 선이 두 덩어리 사이를 지나갑니다. **이 선이 모델이 학습한 전부**입니다 —
`w1`, `w2`, `b` 세 숫자가 이 선 하나를 정합니다.

> **여기가 5주차로 넘어가는 다리입니다.**
> `nn.Linear` 하나는 **직선 하나**밖에 못 긋습니다.
> 두 덩어리가 섞여 있거나 원 모양으로 배치되어 있으면 직선으로는 절대 못 가릅니다.
> 그래서 **층을 쌓고 그 사이에 활성함수를 넣습니다** — 그게 다음 주차 신경망(MLP)입니다.

---

### 이 노트북 체크리스트

- [ ] `nn.Linear(2, 1)` 인 이유를 shape 으로 설명할 수 있다
- [ ] 학습 루프가 실습 6과 **똑같다**는 것을 확인했다
- [ ] `BCEWithLogitsLoss` 에 시그모이드를 따로 붙이지 않는 이유를 안다
- [ ] 정확도가 90% 이상 나왔다
- [ ] 결정 경계를 그려 보고 **직선 하나의 한계**를 말할 수 있다